# Modélisation – [Hypothèse : ex. H3b, logement et besoins des ménages]
**Contributeurs :** [prénom] · **Date :** [jj/mm/2026]

Template commun de l'équipe. Il suit le déroulé de la feuille de route (partie 1.4) et passe par `commun.py` pour tout ce qui doit être identique : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`.

À adapter pour chaque hypothèse : la question, les variables du bloc (section 0) et les modèles (section 4). Le reste ne change pas.

**Question :** [la question de la fiche de l'hypothèse]

## 0. Paramètres de l'hypothèse
- `HYPOTHESE` : préfixe du nom des modèles dans le tableau commun (« H3b – Ridge – parc + vacance »).
- `VARIABLES_BLOC` : les variables testées, ajoutées au prix N-1.
- `VARIABLES_LOG` : les variables asymétriques passées en log, pour les seuls modèles qui en ont besoin (liste de l'audit complémentaire).

In [ ]:
HYPOTHESE = "H3b"
VARIABLES_BLOC = ["taux_vacance", "part_avant1946"]     # EXEMPLE : à remplacer par les variables du bloc
VARIABLES_LOG = []                                      # à remplir avec la liste de l'audit complémentaire

## 1. Import et chargement
`charger_table()` retire une seule fois les lignes à case vide (masque commun) : tous les modèles et la naïve sont jugés sur les mêmes lignes. `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec l'apprentissage sur les années d'avant.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from copy import deepcopy

import commun
from commun import CIBLE, charger_table, fenetres, plis_temporels, evaluer, tester
from gabarit_hierarchique import ModeleHierarchique

table = charger_table()
FENETRES = fenetres(table)

absentes = sorted(set(VARIABLES_BLOC + VARIABLES_LOG) - set(table.columns))
assert not absentes, f"Variables absentes de la table : {absentes}"
print("Table :", table.shape)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

## 2. Constat rapide
Distribution des variables du bloc et lien avec le prix (rho de Spearman), sur les années d'apprentissage de la première fenêtre seulement : l'année jugée n'est jamais regardée avant la modélisation.

In [ ]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_constat[VARIABLES_BLOC].describe().round(2).to_string())

liens = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        rho, p = stats.spearmanr(e[v], e[CIBLE])
        liens.append({"type de bien": type_bien, "variable": v, "rho": rho, "p-value": p})
pd.DataFrame(liens).pivot(index="variable", columns="type de bien", values="rho").round(2)

## 3. Fonctions du notebook
`lancer` fait tourner un modèle sur toutes les fenêtres, un modèle par type de bien, et ajoute ses lignes aux tableaux communs (`evaluer`, `tester`).
- **Pipeline** : mise à l'échelle puis modèle ; les paramètres de la mise à l'échelle sont appris sur l'apprentissage seulement.
- **`log=True`** : les `VARIABLES_LOG` et la cible passent en log ; la prévision est remise en €/m² avant `evaluer` (décision d'équipe 1.6). À réserver aux modèles qui en ont besoin (régressions, Ridge / Lasso / Elastic Net), jamais aux arbres.
- **`grille`** : réglage par `GridSearchCV` sur les plis temporels de `commun.py` (une année de validation par pli, jamais l'année jugée).
- **Modèle hiérarchique** (`ModeleHierarchique` de `gabarit_hierarchique.py`) : pas de pipeline (il met lui-même les variables à l'échelle) ni de grille ; il lit le sous-bassin dans l'index, que `lancer` conserve. Avec `log=True`, il apprend sur le log du prix et ses prévisions sont remises en €/m².

In [ ]:
resultats, tests, ajustes = [], [], {}


def en_log(X):
    """Variables asymétriques en log (log(1 + x) : garde les zéros)."""
    X = X.copy()
    cols = [c for c in VARIABLES_LOG if c in X.columns]
    X[cols] = np.log1p(X[cols])
    return X


class Hierarchique(RegressorMixin, BaseEstimator):
    """Enveloppe scikit-learn du modèle hiérarchique : log de la cible si demandé, prévisions en €/m²."""
    def __init__(self, modele=None, log=False):
        self.modele = modele
        self.log = log

    def fit(self, X, y):
        self.modele_ = deepcopy(self.modele).fit(X, np.log(y) if self.log else y)
        return self

    def predict(self, X):
        p = np.asarray(self.modele_.predict(X))
        return np.exp(p) if self.log else p


def lancer(nom, modele, variables, log=False, grille=None):
    """Ajuste un modèle par fenêtre et par type de bien, puis evaluer et tester."""
    nom = f"{HYPOTHESE} – {nom}"
    for annee, (app, ev) in FENETRES.items():
        prevision = pd.Series(np.nan, index=ev.index)
        for type_bien, e in app.groupby("type_bien"):
            idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
            X_app, X_ev = e[variables], ev.loc[idx, variables]
            if isinstance(modele, ModeleHierarchique):
                assert not grille, "pas de grille pour le modèle hiérarchique"
                if log:
                    X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = Hierarchique(modele, log).fit(X_app, e[CIBLE])
                prevision.loc[idx] = m.predict(X_ev)
                ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
                continue
            m = Pipeline([("echelle", StandardScaler()), ("modele", clone(modele))])
            prefixe = "modele__"
            if log:
                X_app, X_ev = en_log(X_app), en_log(X_ev)
                m = TransformedTargetRegressor(regressor=m, func=np.log, inverse_func=np.exp)
                prefixe = "regressor__modele__"
            if grille:
                # plis recalculés sur e (positions dans e, même ordre que X_app)
                m = GridSearchCV(m, {prefixe + k: v for k, v in grille.items()},
                                 cv=plis_temporels(e), scoring="neg_mean_absolute_error")
            m.fit(X_app, e[CIBLE])
            prevision.loc[idx] = m.predict(X_ev)
            ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
        assert prevision.notna().all(), f"{nom} : prévisions manquantes"
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)
    return nom

## 4. Modèles
Du plus simple au plus complexe ; chaque modèle répond à une limite du précédent. Tous sont gardés, y compris ceux qui perdent.

### 4.1 Référence : prix N-1 seul
La naïve (prix N = prix N-1) est déjà dans chaque ligne du tableau (« MAE naïve »). Le modèle « prix N-1 seul » est la référence que le bloc doit battre.

In [ ]:
REFERENCE = lancer("Régression linéaire – prix N-1 seul", LinearRegression(), ["prix_n1"])

### 4.2 Régression linéaire : prix N-1 + bloc

In [ ]:
lancer("Régression linéaire – prix N-1 + bloc", LinearRegression(), ["prix_n1"] + VARIABLES_BLOC)

### 4.3 Lasso : quelles variables du bloc survivent
`alpha` réglé sur les plis temporels.

In [ ]:
LASSO = lancer("Lasso – prix N-1 + bloc", Lasso(max_iter=10_000), ["prix_n1"] + VARIABLES_BLOC,
               grille={"alpha": [0.1, 1, 10, 100]})

### 4.4 Gradient boosting
Modèle provisoire de scikit-learn, à remplacer par le gabarit gradient boosting de l'équipe (P3) quand il sera prêt. Pas de log pour les arbres.

In [ ]:
lancer("Gradient boosting – prix N-1 + bloc", HistGradientBoostingRegressor(random_state=commun.GRAINE),
       ["prix_n1"] + VARIABLES_BLOC)

### 4.5 (option) Modèle hiérarchique
Une ordonnée à l'origine propre à chaque sous-bassin (`gabarit_hierarchique.py`). À garder seulement si l'hypothèse le prévoit : avec `prix_n1`, l'effet du territoire est presque nul (ICC proche de 0, voir `test.py`). Retirer le `#` pour le lancer.

In [ ]:
# lancer("Hiérarchique – prix N-1 + bloc", ModeleHierarchique(), ["prix_n1"] + VARIABLES_BLOC)

## 5. Mesures et tests
- **`resultats`** : MAE en €/m² et en % du prix, MAE de la naïve, gain sur la naïve, part des sous-bassins où le modèle gagne.
- **`tests`** : écart de MAE avec la naïve et son intervalle de confiance à 95 % (bootstrap par bassin de vie), Wilcoxon apparié et rang-bisériale.

In [ ]:
res = pd.DataFrame(resultats)
res.round(2)

In [ ]:
tst = pd.DataFrame(tests)
tst.round(3)

### Règle de décision (1.5)
Un modèle est retenu si, sur la majorité des années jugées : l'intervalle du bootstrap est entièrement sous 0, la rang-bisériale est négative et le gain sur la naïve atteint 3 %.

In [ ]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la naïve (décision d'équipe 1.6)

cles = ["modèle", "type de bien", "année jugée"]
regle = res[cles + ["gain sur la naïve (%)"]].merge(tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                          & (regle["gain sur la naïve (%)"] >= SEUIL_GAIN))
decision = regle.groupby(["modèle", "type de bien"])["année réussie"].agg(["sum", "count"])
decision["retenu"] = decision["sum"] > decision["count"] / 2
decision.rename(columns={"sum": "années réussies", "count": "années jugées"})

## 6. Interprétation
### 6.1 Importance des variables
Importance par permutation, sur l'année jugée : de combien la MAE augmente quand on mélange une variable. Choisir le modèle, l'année jugée et le type de bien.

In [ ]:
choix = (LASSO, 2024, "maison")        # à adapter
m, X_ev, y_ev = ajustes[choix]
imp = permutation_importance(m, X_ev, y_ev, scoring="neg_mean_absolute_error",
                             n_repeats=10, random_state=commun.GRAINE)
pd.Series(imp.importances_mean, index=X_ev.columns, name="hausse de la MAE (€/m²)").sort_values(ascending=False).round(1)

### 6.2 Analyse des erreurs selon le contrôle qualité
L'erreur est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle : groupe « part nulle », puis quartiles parmi les parts non nulles ; MAE et erreur moyenne signée par groupe, et rho de Spearman entre l'erreur absolue et la variable.

In [ ]:
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]

m, X_ev, y_ev = ajustes[choix]
erreur = pd.Series(m.predict(X_ev), index=X_ev.index) - y_ev       # signée : positive = prix surestimé
controle = table.loc[X_ev.index, CONTROLE]

for v in CONTROLE:
    x = controle[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = erreur.groupby(groupe).agg(MAE=lambda s: s.abs().mean(), erreur_signee="mean", lignes="size")
    rho, p = stats.spearmanr(x, erreur.abs())
    print(f"\n{v} : rho = {rho:.2f} (p = {p:.3f})")
    print(tableau.round(1).to_string())

## 7. Conclusion
- Bloc retenu ou non, par type de bien, selon la règle de décision, avec les chiffres (gain, intervalle, rang-bisériale).
- Variables qui portent l'effet (Lasso, importance).
- Erreurs concentrées ou non dans les territoires aux données estimées (limite à déclarer).
- Conclusion transmise à Chauve-Souris pour le modèle complet.